### Q1 - Building the knowledge base

Last two digits of my roll number 1024170424 are 2 and 4.

d = 2 -> category[2 % 3] = category[2] = general
d = 4 -> category[4 % 3] = category[1] = account

so my two personalized entries are one general question and one account question.

In [1]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

personal_entries = [
    {"question": "what are your contact details", "answer": "You can email us at support@company.com or call our helpline.", "keywords": "contact email helpline", "category": "general"},
    {"question": "how do i update my registered mobile number", "answer": "Go to Profile > Edit Details > Mobile Number to update it.", "keywords": "mobile number update", "category": "account"},
]

faq_df = pd.DataFrame(fixed_entries + personal_entries)
faq_df

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,what are your contact details,You can email us at support@company.com or cal...,contact email helpline,general
5,how do i update my registered mobile number,Go to Profile > Edit Details > Mobile Number t...,mobile number update,account


### Q2 - Scoring function

For a given query I split it into words and check how many of those words show up in the question, answer and keywords of each entry. Higher count = higher confidence match. Then I sort everything by score.

In [2]:
def get_matches(query, df):
    query_words = query.lower().split()
    scores = []
    for i in range(len(df)):
        row = df.iloc[i]
        text = (row["question"] + " " + row["answer"] + " " + row["keywords"]).lower()
        count = 0
        for w in query_words:
            if w in text:
                count = count + 1
        scores.append(count)
    result = df.copy()
    result["score"] = scores
    result = result[result["score"] > 0]
    result = result.sort_values(by="score", ascending=False)
    return result

get_matches("how can i pay my fee", faq_df)

,question,answer,keywords,category,score
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,5
5,how do i update my registered mobile number,Go to Profile > Edit Details > Mobile Number t...,mobile number update,account,3
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,2
4,what are your contact details,You can email us at support@company.com or cal...,contact email helpline,general,2
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general,1


### Q3 - Entries by category

Calling it with "account" since that is one of the categories from my personalized entries in Q1.

In [3]:
def same_category(category_name, df):
    matches = df[df["category"] == category_name]
    return matches["question"]

same_category("account", faq_df)

,question
1,how to reset password
5,how do i update my registered mobile number


### Q4 - Add a new keyword

I picked the first entry (annual fee) and I am adding a keyword the user types in.

In [4]:
new_keyword = input("Enter a new keyword for the annual fee entry: ")
faq_df.loc[0, "keywords"] = faq_df.loc[0, "keywords"] + " " + new_keyword
faq_df.to_csv("1024170424_faq_data.csv", index=False)
faq_df

Enter a new keyword for the annual fee entry: 55


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge 55,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,what are your contact details,You can email us at support@company.com or cal...,contact email helpline,general
5,how do i update my registered mobile number,Go to Profile > Edit Details > Mobile Number t...,mobile number update,account


### Q5 - Count of entries per category

In [5]:
category_counts = faq_df.groupby("category").size()
category_counts

,0
category,
account,2
billing,2
general,2


### Q6 - Handling ties

I changed the scoring function so that after finding the top score, it checks if more than one entry shares that top score. If yes, it prints all of them instead of just picking the first one.

Query "fee" matches both fee related entries with the same score so that produces a tie.
Query "reset password" only matches one entry strongly, so that one does not tie.

In [6]:
def get_matches_v2(query, df):
    query_words = query.lower().split()
    scores = []
    for i in range(len(df)):
        row = df.iloc[i]
        text = (row["question"] + " " + row["answer"] + " " + row["keywords"]).lower()
        count = 0
        for w in query_words:
            if w in text:
                count = count + 1
        scores.append(count)
    result = df.copy()
    result["score"] = scores
    result = result[result["score"] > 0]
    if len(result) == 0:
        print("no matches found")
        return result
    top_score = result["score"].max()
    tied = result[result["score"] == top_score]
    if len(tied) > 1:
        print("tie found, all top matching entries:")
        print(tied[["question", "score"]])
    else:
        print("single best match:")
        print(tied[["question", "score"]])
    return result.sort_values(by="score", ascending=False)

In [7]:
get_matches_v2("fee", faq_df)

tie found, all top matching entries:
                 question  score
0  what is the annual fee      1
3   how can i pay the fee      1


,question,answer,keywords,category,score
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge 55,billing,1
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,1


In [8]:
get_matches_v2("reset password", faq_df)

single best match:
                question  score
1  how to reset password      2


,question,answer,keywords,category,score
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
